# ARGOS Week 2 — EDA & Data Quality

**목적**: 합성 생성된 3년치 ICN 허브 운항 데이터의 탐색적 분석 및 품질 검증.
Week 3 LightGBM 지연 예측 모델 학습 전 데이터 신뢰성 확인이 목적입니다.

| | |
|---|---|
| 노선 수 | 60 (ICN 출발) |
| 기간 | 2022-01-01 ~ 2024-12-31 (3년) |
| 기재 | B737-800, A321neo, B777-300ER, B787-9, B747-8i |
| 지연 코드 | IATA AHM 730 (2자리, 00-99) |

In [ ]:
import sys
sys.path.insert(0, '../src')

import duckdb
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

DB_PATH = '../data/db/argos.duckdb'
con = duckdb.connect(DB_PATH, read_only=True)
print(f'Connected to {DB_PATH}')
print('Tables:', [t[0] for t in con.execute('SHOW TABLES').fetchall()])

## 1. 데이터 개요

In [ ]:
overview = con.execute("""
SELECT
    COUNT(*)                                                        AS total_flights,
    COUNT(DISTINCT route_id)                                        AS routes,
    COUNT(DISTINCT aircraft_registration)                           AS aircraft,
    MIN(scheduled_dep_utc)::DATE                                    AS start_date,
    MAX(scheduled_dep_utc)::DATE                                    AS end_date,
    ROUND(AVG(CASE WHEN status != 'CNX' THEN dep_delay_minutes END), 1) AS avg_delay_min,
    ROUND(SUM(CASE WHEN dep_delay_minutes <= 15 AND status != 'CNX' THEN 1.0 ELSE 0 END)
          / NULLIF(SUM(CASE WHEN status != 'CNX' THEN 1 ELSE 0 END), 0) * 100, 1) AS otp_d15_pct,
    ROUND(SUM(CASE WHEN status = 'CNX' THEN 1.0 ELSE 0 END) / COUNT(*) * 100, 2)  AS cancel_pct
FROM flights
""").df()

display(overview.T.rename(columns={0: 'Value'}))

In [ ]:
# OTP D+15 by region
otp_region = con.execute("""
SELECT
    r.region,
    COUNT(*)                                                AS flights,
    ROUND(AVG(f.dep_delay_minutes), 1)                      AS avg_delay_min,
    ROUND(SUM(CASE WHEN f.dep_delay_minutes <= 15 THEN 1.0 ELSE 0 END)
          / COUNT(*) * 100, 1)                              AS otp_d15_pct
FROM flights f
JOIN routes r ON r.route_id = f.route_id
WHERE f.status != 'CNX'
GROUP BY r.region
ORDER BY otp_d15_pct
""").df()

fig = px.bar(
    otp_region, x='region', y='otp_d15_pct',
    color='otp_d15_pct', color_continuous_scale='RdYlGn',
    range_color=[60, 95],
    text='otp_d15_pct',
    title='지역별 D+15 정시율 (OTP %)',
    labels={'otp_d15_pct': 'OTP D+15 (%)', 'region': '지역'},
)
fig.add_hline(y=78, line_dash='dash', line_color='navy',
              annotation_text='KE 목표 78%', annotation_position='right')
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(height=420, showlegend=False)
fig.show()

## 2. 지연 분포 분석

In [ ]:
# Delay distribution (operated, delayed flights only)
delays = con.execute("""
SELECT dep_delay_minutes
FROM flights
WHERE status != 'CNX' AND dep_delay_minutes > 0 AND dep_delay_minutes < 360
""").df()

fig = px.histogram(
    delays, x='dep_delay_minutes', nbins=60,
    title='지연 시간 분포 (지연 발생 항공편, 0~360분)',
    labels={'dep_delay_minutes': '지연 시간 (분)', 'count': '편수'},
    color_discrete_sequence=['#003087'],  # Korean Air navy
)
fig.add_vline(x=15, line_dash='dash', line_color='red',
              annotation_text='D+15 기준', annotation_position='top right')
fig.add_vline(x=delays['dep_delay_minutes'].mean(), line_dash='dot', line_color='orange',
              annotation_text=f"평균 {delays['dep_delay_minutes'].mean():.0f}분")
fig.update_layout(height=380)
fig.show()

print(delays['dep_delay_minutes'].describe().round(1))

In [ ]:
# Monthly avg delay heatmap (route region × month)
monthly = con.execute("""
SELECT
    r.region,
    EXTRACT(MONTH FROM f.scheduled_dep_utc)::INTEGER AS month,
    ROUND(AVG(f.dep_delay_minutes), 1)               AS avg_delay
FROM flights f
JOIN routes r ON r.route_id = f.route_id
WHERE f.status != 'CNX'
GROUP BY r.region, month
ORDER BY r.region, month
""").df()

pivot = monthly.pivot(index='region', columns='month', values='avg_delay')
pivot.columns = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

fig = px.imshow(
    pivot, text_auto='.1f', aspect='auto',
    color_continuous_scale='YlOrRd',
    title='지역 × 월별 평균 지연 (분) — 계절성 패턴',
    labels={'color': '평균 지연 (분)'},
)
fig.update_layout(height=420)
fig.show()

In [ ]:
# Hourly departure pattern (KST = UTC+9)
hourly = con.execute("""
SELECT
    (EXTRACT(HOUR FROM scheduled_dep_utc)::INTEGER + 9) % 24 AS dep_hour_kst,
    COUNT(*)                                                  AS flights,
    ROUND(AVG(dep_delay_minutes), 1)                          AS avg_delay
FROM flights
WHERE status != 'CNX'
GROUP BY dep_hour_kst
ORDER BY dep_hour_kst
""").df()

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Bar(
    x=hourly['dep_hour_kst'], y=hourly['flights'],
    name='편수', marker_color='#003087', opacity=0.6,
), secondary_y=False)
fig.add_trace(go.Scatter(
    x=hourly['dep_hour_kst'], y=hourly['avg_delay'],
    name='평균 지연 (분)', mode='lines+markers',
    line=dict(color='crimson', width=2),
), secondary_y=True)
fig.update_layout(
    title='시간대별 출발 편수 및 평균 지연 (KST)',
    xaxis_title='출발 시각 (KST)', height=380,
)
fig.update_yaxes(title_text='편수', secondary_y=False)
fig.update_yaxes(title_text='평균 지연 (분)', secondary_y=True)
fig.show()

## 3. 노선별 분석

In [ ]:
# Top 15 most delayed routes
route_delay = con.execute("""
SELECT
    f.route_id,
    r.dest_name,
    r.region,
    COUNT(*)                                                      AS flights,
    ROUND(AVG(f.dep_delay_minutes), 1)                            AS avg_delay,
    ROUND(SUM(CASE WHEN f.dep_delay_minutes <= 15 THEN 1.0 ELSE 0 END)
          / COUNT(*) * 100, 1)                                    AS otp_pct
FROM flights f
JOIN routes r ON r.route_id = f.route_id
WHERE f.status != 'CNX'
GROUP BY f.route_id, r.dest_name, r.region
ORDER BY avg_delay DESC
LIMIT 15
""").df()

fig = px.bar(
    route_delay, x='avg_delay', y='route_id',
    orientation='h', color='region',
    hover_data=['dest_name', 'flights', 'otp_pct'],
    title='평균 지연 상위 15개 노선',
    labels={'avg_delay': '평균 지연 (분)', 'route_id': '노선'},
)
fig.update_layout(height=480, yaxis={'categoryorder': 'total ascending'})
fig.show()

## 4. IATA 지연 코드 분석

In [ ]:
# IATA delay code frequency + responsibility
codes = con.execute("""
SELECT
    f.delay_code,
    dc.description,
    f.delay_responsibility,
    COUNT(*) AS cnt
FROM flights f
JOIN delay_codes_ref dc ON dc.code = f.delay_code
WHERE f.delay_code IS NOT NULL
GROUP BY f.delay_code, dc.description, f.delay_responsibility
ORDER BY cnt DESC
LIMIT 20
""").df()

# Group label: code range category
def code_category(code):
    if code is None: return 'Unknown'
    c = int(code)
    if c < 10: return '00-09 Passenger'
    if c < 20: return '10-19 Cargo'
    if c < 30: return '20-29 Handling'
    if c < 40: return '30-39 Technical'
    if c < 50: return '40-49 Damage'
    if c < 60: return '50-59 Operations'
    if c < 70: return '60-69 Weather'
    if c < 80: return '70-79 ATC'
    if c < 90: return '80-89 Airport'
    return '90-99 Reactionary'

codes['category'] = codes['delay_code'].apply(code_category)

fig = px.treemap(
    codes, path=['category', 'delay_code'], values='cnt',
    hover_data=['description'],
    title='IATA AHM 730 지연 코드 분포 (상위 20개)',
    color='cnt', color_continuous_scale='Blues',
)
fig.update_layout(height=500)
fig.show()

# Responsibility breakdown
resp = con.execute("""
SELECT delay_responsibility, COUNT(*) AS cnt
FROM flights
WHERE delay_responsibility IS NOT NULL
GROUP BY 1 ORDER BY cnt DESC
""").df()
resp['label'] = resp['delay_responsibility'].map({'A':'항공사(A)','ATC':'ATC','W':'기상(W)','AP':'공항(AP)'})
fig2 = px.pie(resp, values='cnt', names='label', title='지연 책임 분류', hole=0.4)
fig2.update_layout(height=340)
fig2.show()

## 5. 기재 (Fleet) 분석

In [ ]:
fleet = con.execute("""
SELECT
    aircraft_type,
    COUNT(*)                                                      AS flights,
    COUNT(DISTINCT aircraft_registration)                         AS aircraft,
    ROUND(AVG(dep_delay_minutes), 1)                              AS avg_delay,
    ROUND(SUM(CASE WHEN dep_delay_minutes <= 15 THEN 1.0 ELSE 0 END)
          / COUNT(*) * 100, 1)                                    AS otp_pct,
    ROUND(AVG(load_factor) * 100, 1)                              AS avg_lf_pct,
    ROUND(AVG(distance_nm))                                       AS avg_dist_nm
FROM flights
WHERE status != 'CNX'
GROUP BY aircraft_type
ORDER BY flights DESC
""").df()

display(fleet)

fig = make_subplots(rows=1, cols=2, subplot_titles=['기재별 운항 편수', '기재별 평균 탑승률'])
fig.add_trace(go.Bar(x=fleet['aircraft_type'], y=fleet['flights'],
                     marker_color='#003087', name='편수'), row=1, col=1)
fig.add_trace(go.Bar(x=fleet['aircraft_type'], y=fleet['avg_lf_pct'],
                     marker_color='#E02020', name='탑승률(%)'), row=1, col=2)
fig.update_layout(height=370, showlegend=False, title_text='기재 운용 현황')
fig.show()

## 6. 탑승률 (Load Factor) 분석

In [ ]:
lf_data = con.execute("""
SELECT load_factor, dep_delay_minutes, aircraft_type, distance_nm
FROM flights
WHERE status != 'CNX' AND load_factor > 0
USING SAMPLE 10000
""").df()

fig = px.scatter(
    lf_data, x='load_factor', y='dep_delay_minutes',
    color='aircraft_type', opacity=0.3,
    trendline='lowess',
    title='탑승률 vs 출발 지연 (샘플 10,000편)',
    labels={'load_factor': '탑승률', 'dep_delay_minutes': '출발 지연 (분)'},
    range_y=[0, 120],
)
fig.update_layout(height=420)
fig.show()

# Monthly load factor trend
lf_monthly = con.execute("""
SELECT
    DATE_TRUNC('month', scheduled_dep_utc)::DATE AS month,
    ROUND(AVG(load_factor) * 100, 1)             AS avg_lf_pct
FROM flights
WHERE status != 'CNX'
GROUP BY 1 ORDER BY 1
""").df()

fig2 = px.line(lf_monthly, x='month', y='avg_lf_pct',
               title='월별 평균 탑승률 추이',
               labels={'month': '월', 'avg_lf_pct': '탑승률 (%)'})
fig2.add_hline(y=82, line_dash='dash', line_color='green', annotation_text='설계 기준 82%')
fig2.update_layout(height=320)
fig2.show()

## 7. 데이터 품질 검증 (Validator)

In [ ]:
from argos.data_gen.validator import DataValidator

validator = DataValidator(DB_PATH)
report = validator.run_all()
report.print_summary()

# Summary for notebook output
import pandas as pd
results_df = pd.DataFrame([
    {'check': r.name, 'passed': r.passed, 'severity': r.severity,
     'affected_rows': r.affected_rows, 'message': r.message}
    for r in report.results
])
display(results_df.style.apply(
    lambda row: ['background: #d4edda' if row['passed'] else
                 'background: #fff3cd' if row['severity'] == 'warning' else
                 'background: #f8d7da'] * len(row),
    axis=1
))

## 8. ML 피처 사전 검토 (Week 3 준비)

LightGBM 지연 예측 모델에 사용할 피처의 분포와 결측 여부를 사전 확인합니다.

In [ ]:
# Feature completeness for ML
features = [
    'dep_hour_utc', 'dep_month', 'dep_dow',  # generated columns
    'distance_nm', 'block_time_minutes',
    'load_factor', 'cargo_kg', 'fuel_uplift_kg',
    'dep_delay_minutes',  # target
]

feature_stats = []
for col in features:
    row = con.execute(f"""
        SELECT
            '{col}'               AS feature,
            COUNT(*)              AS total,
            SUM(CASE WHEN {col} IS NULL THEN 1 ELSE 0 END) AS nulls,
            ROUND(MIN({col}), 2)  AS min_val,
            ROUND(AVG({col}), 2)  AS mean_val,
            ROUND(MAX({col}), 2)  AS max_val
        FROM flights WHERE status != 'CNX'
    """).df()
    feature_stats.append(row)

feat_df = pd.concat(feature_stats, ignore_index=True)
feat_df['null_pct'] = (feat_df['nulls'] / feat_df['total'] * 100).round(2)
display(feat_df[['feature', 'total', 'null_pct', 'min_val', 'mean_val', 'max_val']])

print('\n✅ Week 3 ML 준비 상태:')
if feat_df['null_pct'].max() < 5:
    print('  피처 결측률 < 5% — 학습 데이터 준비 완료')
else:
    high_null = feat_df[feat_df['null_pct'] >= 5]['feature'].tolist()
    print(f'  결측률 주의 피처: {high_null}')

con.close()